
# FISTA with a TV proximal step

FISTA, which reaches a TV-regularised solution faster than gradient methods.

Source: [Python/demos/d20_Algorithms05.py](https://github.com/CERN/TIGRE/blob/master/Python/demos/d20_Algorithms05.py)_ · [MATLAB/Demos/d20_Algorithms05.m](https://github.com/CERN/TIGRE/blob/master/MATLAB/Demos/d20_Algorithms05.m)_

.. dropdown:: MATLAB version of this demo
   :icon: code

   .. literalinclude:: /../MATLAB/Demos/d20_Algorithms05.m
      :language: matlab


## Initialize



In [ ]:
import tigre
import numpy as np
from tigre.utilities import sample_loader
from tigre.utilities import CTnoise
import tigre.algorithms as algs
from matplotlib import pyplot as plt

## Geometry



In [ ]:
geo = tigre.geometry_default(high_resolution=False)

## Load data and generate projections



In [ ]:
# define angles
angles = np.linspace(0, 2 * np.pi, 100, endpoint=False)
# Load thorax phantom data
head = sample_loader.load_head_phantom(geo.nVoxel)
# generate projections
projections = tigre.Ax(head, geo, angles)
# add noise
noise_projections = CTnoise.add(projections, Poisson=1e5, Gaussian=np.array([0, 10]))

## FISTA is a quadratically converging algorithm.

# 'hyper': This parameter should approximate the largest
#          eigenvalue in the A matrix in the equation Ax-b and Atb.
#          Empirical tests show that for, the headphantom object:
#
#               geo.nVoxel = [64,64,64]'    ,      hyper (approx=) 2.e8
#               geo.nVoxel = [512,512,512]' ,      hyper (approx=) 2.e4
#          Default: 2.e8
# 'tviter':  Number of iterations of Im3ddenoise to use. Default: 20
# 'lambda':  Multiplier for the tvlambda used, which is proportional to
#            L (hyper). Default: 0.1
# 'verbose': get feedback or not. Default: 1
# --------------------------------------------------------------------------
imgFISTA_default = algs.fista(noise_projections, geo, angles, 100)

## Adding a different hyper parameter
imgFISTA_hyper = algs.fista(noise_projections, geo, angles, 100, hyper=2.0e6)

## Recon more significant tv parameters
imgFISTA_hightv = algs.fista(
    noise_projections, geo, angles, 100, hyper=2.0e6, tviter=100, tvlambda=20
)

## Plot results
tigre.plotimg(
    np.concatenate([imgFISTA_default, imgFISTA_hyper, imgFISTA_hightv], axis=1),
    dim="z",
    clims=[0, 1],
)